# Where does a complex Newton step take its starting point?

A single cubic has three equally simple roots. Derive the iteration, colour starting values by destination and use a separate convergence history to distinguish a bad first step from slow local convergence.

## Continue the tangent step into the complex plane

A holomorphic function has a complex linear approximation $f(z+s)\approx f(z)+f'(z)s$. Requiring this approximation to vanish gives $s=-f(z)/f'(z)$ wherever the derivative is nonzero. For $f(z)=z^3-1$, the resulting map is

$$N(z)=\frac{2z}{3}+\frac{1}{3z^2}.$$

The second term is a pole at zero. Starting at a small positive number sends the next iterate far to the right, after which large iterates return approximately by multiplication by 2/3. At a root $r$, write $z=r+e$ and substitute $r^3=1$. Exact cancellation of the linear error gives

$$
N(r+e)-r=e^2(3r+2e)/[3(r+e)^2].
$$

Sufficiently near $r$, the new error is about $e^2/r$. This argument predicts the speed after entry into a local neighbourhood. It has no information about which root a distant initial point will reach.

Let $s=\exp(2\pi i/3)$. Since $s^3=1$, symmetry gives

$$
N(sz)=sN(z).
$$

The entire sequence, including its root, rotates with the start. Conjugate starting points likewise produce conjugate histories because the coefficients are real. These are two independent checks for numerical basin maps. We classify a pixel only once its residual falls below a declared tolerance. A starting pole and a trajectory that exhausts its iteration allowance can both appear unclassified. Increasing the allowance separates these cases for many pixels.

## Follow a detour before quadratic convergence

From $z_0=1/2$ the first iterate is $1/3+4/3=5/3$. The next is $10/9+3/25=277/225\approx1.23111$. The error initially rises from 0.5 to 0.667. By contrast, starting at 0.1 gives $z_1=33.4$. The subsequent near-geometric return explains the long shoulder in the second figure. Both starts ultimately approach the real root.

Move the orange sample by writing a complex number such as $-0.5+0.85i$ in Python as `-0.5+0.85j`. The sample mark and the residual trace follow the same new starting point. The intricate boundaries in the image are computed from independent point trajectories. Refining the grid examines finer samples of those boundaries; the speed on the residual graph belongs to one particular start.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
sample_start = 0.1 + 0j
grid_size = 201
max_steps = 38
tolerance = 1.e-8

In [ ]:
assert 31 <= grid_size <= 501 and 1 <= max_steps <= 180 and 0 < tolerance < 0.1
roots = np.exp(2j*np.pi*np.arange(3)/3)
def step(z):
    return (2*z + 1/(z*z))/3

def classify(initial):
    z = np.asarray(initial, dtype=complex).copy().reshape(-1)
    labels = np.full(z.size, -1, dtype=int)
    active = np.ones(z.size, dtype=bool)
    for _ in range(max_steps+1):
        with np.errstate(over="ignore",invalid="ignore",divide="ignore"):
            residual = np.abs(z**3-1)
            done = active & (residual < tolerance)
            labels[done] = np.argmin(np.abs(z[done,None]-roots), axis=1)
            active &= ~done & np.isfinite(z) & (np.abs(z) > 1.e-120)
            z[active] = step(z[active])
    return labels.reshape(np.shape(initial))

points = np.linspace(-1.6,1.6,grid_size)
X,Y = np.meshgrid(points,points)
basin = classify(X+1j*Y)
assert np.array_equal(classify(roots), [0,1,2]) and classify(0j) == -1
check = 0.75 + 0.1j
assert np.isclose(step(roots[1]*check),roots[1]*step(check))
print("Basin counts [unclassified, real root, upper root, lower root]:",
      np.bincount((basin+1).ravel(),minlength=4))

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
fig, ax = axes("Real part of initial value", "Imaginary part of initial value")
palette = ["#353943","#add8c9","#85aed5","#e9a98e"]
ax.imshow(basin+1, origin="lower", extent=[-1.6,1.6,-1.6,1.6],
          cmap=ListedColormap(palette),vmin=0,vmax=3,interpolation="nearest")
ax.scatter(roots.real,roots.imag,marker="*",c="black",s=90,label="Roots")
ax.scatter(sample_start.real,sample_start.imag,marker="x",c="black",s=50,label="Sample")
handles = [Patch(color=c,label=s) for c,s in zip(palette,["Unclassified","+1","Upper root","Lower root"])]
ax.legend(handles=handles+ax.get_legend_handles_labels()[0],loc="upper left",fontsize=7,ncol=2)
ax.grid(False)
plt.show()

Each grid point starts an independent cubic iteration. Dark grey means the residual criterion was never met within the allowed steps, including the pole at zero.

In [ ]:
fig, ax = axes("Newton step", "Cubic residual magnitude")
for z0,col in zip([0.5+0j,sample_start,-.5+.8j],[teal,orange,blue]):
    z=complex(z0); history=[]
    for _ in range(max_steps+1):
        if abs(z)>1.e95: break
        value=abs(z**3-1); history.append(max(value,1.e-16))
        if value<tolerance or abs(z)<1.e-120: break
        z=step(z)
    ax.semilogy(range(len(history)),history,marker="o",ms=3,color=col,label=f"Start {z0:.3g}")
ax.axhline(tolerance,color="#484848",ls="--",label="Stopping tolerance")
ax.legend(fontsize=8)
plt.show()

The dashed line is the actual residual tolerance. A small initial magnitude can produce a large first residual followed by a long return.

## Exercise · obtain two basin colours for free

Suppose $0.75+0.10i$ reaches the real root in five updates. What are the destinations and update counts after multiplying the start by either non-real cube root of one?

Write the original iterates as $z_k$. By induction $N^k(sz_0)=sz_k$ and $N^k(s^2z_0)=s^2z_k$. The destinations are $s$ and $s^2$. Since $(sz_k)^3-1=z_k^3-1$, every residual and hence the first crossing of the same threshold is unchanged. Differences in a finely sampled plot may arise because the rotated point falls between pixels.

## Exercise · quantify the final two digits

For a real iterate $z=1.01$, estimate the next error by the quadratic approximation and calculate its exact value using the error formula.

The leading term gives $0.01^2=10^{-4}$. The exact error is $10^{-4}(3.02)/(3\times1.01^2)\approx9.8683\times10^{-5}$. The difference is of the next order in the initial error. The small residual after this update is about three times the error because $f'(1)=3$.